# Real-Time Acoustic Kinematics, Hardware Frequency Filtering & Physics Verification
Welcome to **`pynq-sound-localizer`** (Hardware Overlay **`v1.6.0`**).

This notebook demonstrates:
1. **Overlay Initialization & Status Verification** (True simultaneous dual-sampling, $0.00\,\mu\text{s}$ skew).
2. **Real-Time 10-Second Rolling Kinematics Dashboard** ($A(t)$ physical loudness & sub-Hertz pitch $f_0(t)$).
3. **Hardware Spectral Frequency Filtering & IFFT Reconstruction** (`axis_spectral_mask` bandpass $(\omega_0 \pm \Delta\omega)$).
4. **Physical Inverse-Distance Acoustic Law Verification** (Measuring $V_{\text{RMS}}(r) \propto 1/r$ and $I(r) \propto 1/r^2$).
5. **Continuous Flight Recording & Jupyter Audio Playback**.

## 1. Load the Hardware Overlay
Instantiate `MicrophoneArrayOverlay()`. It automatically downloads and configures the verified `v1.6.0` bitstream on the PYNQ-Z2 board.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from pynq_localizer import MicrophoneArrayOverlay, KinematicAnalytics

# Load overlay with default Full-Audio profile (50 kSPS, N=1024)
ol = MicrophoneArrayOverlay()

print(f"✅ Overlay Loaded: {ol.current_profile} profile ({ol.fs_per_ch:.0f} SPS per channel)")
print(f"   Speed of sound at 20°C: {KinematicAnalytics.speed_of_sound(20.0):.2f} m/s")
print(f"   Filter State: {ol.filter}")

## 2. Launch the Real-Time 10-Second Rolling Kinematics Dashboard
Click **`Start Stream`** and make sound near the microphones (or play a pure tone from a smartphone).
- **Row 1:** Physical RMS Voltage Loudness Envelope $A(t)$.
- **Row 2:** Sub-Hertz Dominant Frequency Trajectory $f_0(t)$ ($20\,\text{Hz} - 20\,\text{kHz}$).
- **Tabs:** Toggle between **Mic 1 (A0)**, **Mic 2 (A1)**, and **Dual Overlay**.

In [ ]:
# Launch the live interactive multi-tab instrument
app = ol.kinematics_dashboard()


## 3. Direct Clean Data Handoff in Python
Extract clean, sanitized, non-NaN motion telemetry directly from the dashboard into NumPy arrays for instant physics curve fitting.

In [ ]:
t_clean, amp_clean, freq_clean = app.get_clean_data(channel=1)

print(f"Captured {len(t_clean)} clean telemetry motion points!")
if len(freq_clean) > 0:
    print(f"Observed Frequency Span: {freq_clean.min():.1f} Hz -> {freq_clean.max():.1f} Hz")
    print(f"Max Observed Doppler Shift: {freq_clean.max() - freq_clean.min():.1f} Hz")

## 4. Hardware Spectral Frequency Filtering & IFFT Reconstruction
Configure a hardware bandpass filter around a custom angular frequency range $(\omega_0 \pm \Delta\omega)$ / $(f_0 \pm \Delta f)$ to isolate a specific sound source while rejecting all other interferers in real time.

Then capture all 3 hardware streams concurrently via `ol.capture_all()`:
1. `v_raw_a0`, `v_raw_a1`: Raw interleaved time streams
2. `freqs`, `mags`: Frequency-domain magnitude spectrum
3. `v_filtered`: Reconstructed time-domain audio from the hardware IFFT core (`xfft_1`)

In [ ]:
# 1. Configure Hardware Bandpass Filter (e.g. 1000 Hz +- 150 Hz)
target_f0 = 1000.0
bandwidth = 150.0
ol.filter.set_bandpass(center_hz=target_f0, delta_hz=bandwidth)
print(f"Hardware Filter Configured: {ol.filter}")

# 2. Capture all 3 streams synchronously
v_raw_a0, v_raw_a1, v_filt, freqs, mags = ol.capture_all()

# 3. Plot Raw vs Filtered Time and Frequency Spectrum
t_ms = np.linspace(0, (len(v_raw_a0) / ol.fs_per_ch) * 1000.0, len(v_raw_a0))

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_ms, y=v_raw_a0, mode='lines', name='Raw Mic 1 (A0)', line=dict(color='#FFA500', width=1.2)))
fig.add_trace(go.Scatter(x=t_ms, y=v_filt, mode='lines', name='Filtered Time (IFFT)', line=dict(color='#00FFCC', width=1.8)))
fig.update_layout(template='plotly_dark', title='<b>Hardware Spectral Filtering: Raw vs Filtered Time</b>', xaxis_title='Time (ms)', yaxis_title='Voltage (V)', height=400)
fig.show()

# Reset to bypass after test
ol.filter.bypass()

## 5. Physical Inverse-Distance Law Verification ($1/r$ and $1/r^2$)
Acoustic physical theory dictates that in an open 3D spherical field:
- **Acoustic Pressure Voltage:** $V_{\text{RMS}}(r) \propto \frac{1}{r} = r^{-1.0}$
- **Acoustic Intensity / Energy:** $I(r) \propto \frac{1}{r^2} = r^{-2.0}$

Use `KinematicAnalytics.fit_inverse_distance_law()` to verify this relation experimentally.

In [ ]:
# Example calibration distances in meters and measured RMS voltages
calibrated_distances = np.array([0.25, 0.50, 0.75, 1.00, 1.50, 2.00])
# (Replace with your actual physical measurements)
measured_voltages_rms = 0.85 / calibrated_distances  # Synthetic ideal baseline

fit_results = KinematicAnalytics.fit_inverse_distance_law(calibrated_distances, measured_voltages_rms)

print("===========================================================")
print("  📊 INVERSE-DISTANCE ACOUSTIC REGRESSION RESULTS")
print("===========================================================")
print(f"  • Measured Voltage Exponent (n)  : {fit_results['measured_exponent_n']:.3f} (Ideal = 1.000)")
print(f"  • Measured Intensity Exponent    : {fit_results['intensity_exponent_2n']:.3f} (Ideal = 2.000)")
print(f"  • Goodness of Fit (R²)           : {fit_results['r_squared']:.4f}")
print(f"  • Deviation from Ideal 1/r Law   : {fit_results['error_pct_from_ideal_1_over_r']:.2f}%")
print("===========================================================")

## 6. Continuous Multi-Second Flight Recording & Audio Playback
Record uninterrupted audio to DDR memory and listen to it directly inside Jupyter.

In [ ]:
# Record 3.0 seconds of 50 kSPS dual-channel audio
t_axis, v_mic1, v_mic2 = ol.record_continuous(duration_sec=3.0)

# Listen to Microphone 1
ol.play_audio(channel=1, custom_data=v_mic1)

## 7. Clean Shutdown & Hardware Release

In [ ]:
app.stop()
ol.close()
print("🔒 Hardware resources cleanly released.")